# HEST content extraction and PathFM-to-gene Transformer
Run the notebook from HEST_Local or notebook/; companion modules are in scripts/.


Install dependencies once using README.md.


## 1. Import local companion scripts


In [ ]:
from pathlib import Path
import sys
_here = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (_here, *_here.parents) if (p / 'scripts').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from HEST_Local or its notebook folder.")
if str(PROJECT_ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

from pathlib import Path
import sys, importlib

HELPER_DIR = Path(str(PROJECT_ROOT / 'scripts'))  # Change to your helper-script folder.
sys.path.insert(0, str(HELPER_DIR))
for module_name in ['hest_download_pipeline', 'hest_preprocessing',
                    'hest_content_pipeline', 'hest_transformer_model']:
    if not (HELPER_DIR / f'{module_name}.py').is_file():
        raise FileNotFoundError(f'Missing {HELPER_DIR / (module_name + ".py")}')
    module = importlib.import_module(module_name)
    importlib.reload(module)

from huggingface_hub import login
login()  # Authenticate privately if HEST asks for access.
import hest_content_pipeline as content
import hest_preprocessing as preprocess
import hest_transformer_model as modeling

## 2. Prompt for the data category

In [ ]:
from pathlib import Path
import sys
_here = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (_here, *_here.parents) if (p / 'scripts').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from HEST_Local or its notebook folder.")
if str(PROJECT_ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

from pathlib import Path
METADATA_VERSION = 'HEST_v1_3_0.csv'
DATA_DIR = Path(str(PROJECT_ROOT / 'data' / 'clean_data'))
RUN_DIR = Path(str(PROJECT_ROOT / 'training_run'))
RUN_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

categories = content.get_hest_categories(METADATA_VERSION)
for category, values in categories.items():
    print(f"{category}: {', '.join(values)}")
ORGAN = input('Organ(s), e.g. Breast, or all: ').strip()
SPECIES = input('Species, e.g. Homo sapiens, or all: ').strip()
ST_TECHNOLOGY = input('Technology, e.g. Visium, or all: ').strip()
filtered_metadata = content.query_hest_category(
    organ=ORGAN, species=SPECIES, st_technology=ST_TECHNOLOGY,
    metadata_filename=METADATA_VERSION,
)
# A disk snapshot of the HEST database supports checks after runtime resets.
filtered_metadata.to_csv(RUN_DIR/'hest_reference_metadata.csv', index=False)
display(filtered_metadata.head())
print('Matching slides:', len(filtered_metadata), filtered_metadata['id'].tolist())

## 3. Select slides, download and verify
Enter IDs or a count and 1-based starting position within the filtered cohort. The exact intended cohort is saved as `selected_metadata.csv` before downloading. The checkpoint compares slide IDs and patch-folder counts with the saved manifests. It trusts existing patch contents and does not open their PNG/CSV files. It asks separately whether to download missing slides and redownload slides with missing patches or inconsistent counts. New downloads are validated before being installed.

In [ ]:
from pathlib import Path
import sys
_here = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (_here, *_here.parents) if (p / 'scripts').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from HEST_Local or its notebook folder.")
if str(PROJECT_ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

import pandas as pd

# Reload the category query from disk when needed; rerunning this cell does not
# narrow the available cohort to the previous chunk.
filtered_metadata = pd.read_csv(RUN_DIR/'hest_reference_metadata.csv', dtype={'id': str})
available_ids = filtered_metadata['id'].astype(str).tolist()
download_mode = input('Select slides by (1) ID list or (2) count/range [1/2]: ').strip() or '1'
if download_mode == '1':
    raw_ids = input('Enter IDs separated by commas, e.g. TENX68, TENX69: ')
    selected_ids = list(dict.fromkeys(x.strip() for x in raw_ids.split(',') if x.strip()))
    if not selected_ids:
        raise ValueError('Enter at least one slide ID.')
    unknown_ids = sorted(set(selected_ids) - set(available_ids))
    if unknown_ids:
        raise ValueError(f'IDs absent from the filtered HEST metadata: {unknown_ids}')
elif download_mode == '2':
    n_slides = int(input(f'How many slides? [available {len(available_ids)}]: '))
    start_at = int(input('Start at filtered slide number (1-based) [1]: ') or '1')
    if n_slides < 1 or not 1 <= start_at <= len(available_ids):
        raise ValueError('Enter a positive count and a valid start position.')
    selected_ids = available_ids[start_at - 1:start_at - 1 + n_slides]
    print(f'Positions {start_at} to {start_at+len(selected_ids)-1}: {len(selected_ids)} slides.')
else:
    raise ValueError("Choose '1' for IDs or '2' for count/range.")

selected_metadata = filtered_metadata.set_index('id').loc[selected_ids].reset_index()
RUN_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
selected_metadata.to_csv(RUN_DIR/'selected_metadata.csv', index=False)
selected_metadata.to_csv(DATA_DIR/'selected_metadata.csv', index=False)
selected_metadata[['id']].to_csv(RUN_DIR/'download_selection.csv', index=False)
checkpoint = content.checkpoint_cohort(
    DATA_DIR, RUN_DIR, metadata_filename=METADATA_VERSION,
    verify_contents=False, max_workers=2,
)
display(checkpoint['slides'])

## 4a. Verify checkpoint: slide IDs and counts only
This cell reads the saved selection, counts slide/patch folders and compares them with the saved manifests. It saves and displays the report, then finishes. It does not open patch PNG/CSV contents, ask for input, download/repair data, or split slides. Missing or inconsistent slides are reported without stopping the count checkpoint.


In [ ]:
from pathlib import Path
import sys
_here = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (_here, *_here.parents) if (p / 'scripts').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from HEST_Local or its notebook folder.")
if str(PROJECT_ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

from pathlib import Path
import sys, importlib
import pandas as pd

HELPER_DIR = Path(globals().get('HELPER_DIR', str(PROJECT_ROOT / 'scripts')))
DATA_DIR = Path(globals().get('DATA_DIR', str(PROJECT_ROOT / 'data' / 'clean_data')))
RUN_DIR = Path(globals().get('RUN_DIR', str(PROJECT_ROOT / 'training_run')))
METADATA_VERSION = globals().get('METADATA_VERSION', 'HEST_v1_3_0.csv')
sys.path.insert(0, str(HELPER_DIR))
import hest_content_pipeline as content
importlib.reload(content)

# This cell only compares selected slide IDs and patch-folder counts.
# No downloads, repairs, content validation, or split prompts are run here.
RUN_DIR.mkdir(parents=True, exist_ok=True)
candidates = [RUN_DIR/'selected_metadata.csv', RUN_DIR/'download_selection.csv',
              DATA_DIR/'selected_metadata.csv']
selection_csv = next((p for p in candidates if p.is_file()), None)
if selection_csv is None:
    raise FileNotFoundError('Set the paths to your saved selection CSV and downloaded data.')
reference_csv = RUN_DIR/'hest_reference_metadata.csv'
reference_metadata = pd.read_csv(reference_csv, dtype={'id': str}) if reference_csv.is_file() else None
print('Counting selected slides and patch folders...', flush=True)
checkpoint = content.audit_cohort(
    selection_csv, DATA_DIR, reference_metadata=reference_metadata,
    verify_contents=False,
)
active_ids = {row['slide_id'] for row in checkpoint['records']}
active_metadata = checkpoint['metadata'][checkpoint['metadata']['id'].isin(active_ids)].copy()
print('Saving count report and patch list...', flush=True)
checkpoint['slides'].to_csv(RUN_DIR/'checkpoint_slides.csv', index=False)
checkpoint['patch_issues'].to_csv(RUN_DIR/'checkpoint_patch_issues.csv', index=False)
active_metadata.to_csv(RUN_DIR/'active_metadata.csv', index=False)
content.write_split_manifest(checkpoint['records'], RUN_DIR/'checkpoint_patches.csv')
display(checkpoint['slides'])
print(f"Selected slides present: {checkpoint['present_count']}/{checkpoint['selected_count']}; "
      f"available selected patches: {len(checkpoint['records'])}.")
if checkpoint['extra_slide_ids']:
    print('Unselected slides ignored:', ', '.join(checkpoint['extra_slide_ids']))
print('Checkpoint finished. Run section 4b separately when you want to create the split.')


## 4b. Choose grouping and split
Run this cell separately when you are ready to split. It loads the saved checkpoint patch list without auditing the files again, asks for patient or slide grouping, then asks for the training percentage. Choose `slide` if patient IDs are unavailable. The split is saved for section 5. Only patient grouping requires a completed patient map; slide grouping is labeled as slide-level evaluation.


In [ ]:
from pathlib import Path
import sys
_here = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (_here, *_here.parents) if (p / 'scripts').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from HEST_Local or its notebook folder.")
if str(PROJECT_ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

from pathlib import Path
import sys, importlib
import pandas as pd

HELPER_DIR = Path(globals().get('HELPER_DIR', str(PROJECT_ROOT / 'scripts')))
DATA_DIR = Path(globals().get('DATA_DIR', str(PROJECT_ROOT / 'data' / 'clean_data')))
RUN_DIR = Path(globals().get('RUN_DIR', str(PROJECT_ROOT / 'training_run')))
METADATA_VERSION = globals().get('METADATA_VERSION', 'HEST_v1_3_0.csv')
sys.path.insert(0, str(HELPER_DIR))
import hest_content_pipeline as content
importlib.reload(content)

patch_list_path = RUN_DIR/'checkpoint_patches.csv'
active_metadata_path = RUN_DIR/'active_metadata.csv'
if not patch_list_path.is_file() or not active_metadata_path.is_file():
    raise FileNotFoundError('No saved count checkpoint found. Run section 4a once.')
patch_frame = pd.read_csv(patch_list_path, dtype=str).fillna('')
if patch_frame.empty:
    raise ValueError('The saved checkpoint contains no available patches to split.')
records = patch_frame.drop(columns=['order'], errors='ignore').to_dict('records')
selected_metadata = pd.read_csv(active_metadata_path, dtype={'id': str})
if {row['slide_id'] for row in records} != set(selected_metadata['id']):
    raise ValueError('Checkpoint lists do not match. Rerun section 4a.')
print(f'Loaded {len(selected_metadata)} slides and {len(records)} patch records for splitting.', flush=True)
grouping_unit = input('Split grouping: patient (known IDs) or slide (IDs unavailable) [slide]: ').strip().lower() or 'slide'
if grouping_unit == 'slide':
    group_by_slide = {s: s for s in selected_metadata['id']}
    print('Slide-level split and across-slide variance; patient independence is unverified.')
elif grouping_unit == 'patient':
    patient_map_path = RUN_DIR/'slide_patient_map.csv'
    if not patient_map_path.exists():
        patient_map = content.create_patient_map(selected_metadata, patient_map_path)
    else:
        patient_map = pd.read_csv(patient_map_path, dtype=str).fillna('')
        if not {'slide_id', 'patient_id'}.issubset(patient_map.columns):
            raise ValueError(f'Expected slide_id,patient_id columns in {patient_map_path}')
        new_ids = [s for s in selected_metadata['id'] if s not in set(patient_map['slide_id'])]
        if new_ids:
            patient_map = pd.concat([patient_map, pd.DataFrame({'slide_id': new_ids, 'patient_id': ''})], ignore_index=True)
            patient_map.to_csv(patient_map_path, index=False)
    needs_id = patient_map['slide_id'].isin(selected_metadata['id']) & patient_map['patient_id'].str.strip().eq('')
    if needs_id.any():
        display(patient_map.loc[needs_id])
        raise RuntimeError(f'Fill patient_id in {patient_map_path}, or rerun this cell and choose slide grouping.')
    group_by_slide = content.load_patient_map(patient_map_path, selected_metadata['id'].tolist())
else:
    raise ValueError("Choose 'patient' or 'slide'.")

train_fraction = float(input(f'Percent of {grouping_unit} groups for training [80]: ') or '80') / 100
split = content.save_grouped_split(
    records, group_by_slide, RUN_DIR,
    grouping_unit=grouping_unit, train_fraction=train_fraction,
)
train_records, test_records = split['train_records'], split['test_records']
print(f"{len(split['train_groups'])} train / {len(split['test_groups'])} test {grouping_unit} groups; "
      f'{len(train_records)} / {len(test_records)} patches')

## 5a. Select genes with resumable statistics
Choose start anew or resume before processing. Running sums, counts, panel state and
`gene_variance_all.csv` are saved every 500 scanned patches. Interim variances are
provisional and may be NaN until two training groups are observed. This section
does not generate targets. Resume requires unchanged split, grouping, seed and fraction.
Keep the `.json` and `.npz` checkpoint files in RUN_DIR; the CSV alone cannot resume.


In [ ]:
from pathlib import Path
import sys
_here = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (_here, *_here.parents) if (p / 'scripts').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from HEST_Local or its notebook folder.")
if str(PROJECT_ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

from pathlib import Path
import sys, importlib

HELPER_DIR = Path(globals().get('HELPER_DIR', str(PROJECT_ROOT / 'scripts')))
RUN_DIR = Path(globals().get('RUN_DIR', str(PROJECT_ROOT / 'training_run')))
sys.path.insert(0, str(HELPER_DIR))
import hest_content_pipeline as content
importlib.reload(content)
split = content.load_grouped_split(RUN_DIR)
train_records, test_records = split['train_records'], split['test_records']
records = split['records']
group_by_slide, grouping_unit = split['group_by_slide'], split['grouping_unit']

GENE_STATISTICS_FRACTION = 1 / 3  # Set to 0.2 for one fifth.
GENE_STATISTICS_SEED = 42
mode = content.prompt_checkpoint_mode('Gene statistics / selection')
selected_genes = content.select_top_variable_genes(
    train_records, group_by_slide, top_k=50, all_records=records,
    output_dir=RUN_DIR, grouping_unit=grouping_unit,
    sample_fraction=GENE_STATISTICS_FRACTION, sampling_seed=GENE_STATISTICS_SEED,
    checkpoint_every=500, checkpoint_mode=mode, progress_every=500,
)
if len(selected_genes) != 50:
    raise ValueError(f'Expected 50 genes; found {len(selected_genes)}.')
print('Gene selection complete. Run section 5b separately to generate targets.')
display(selected_genes[:10])


## 5b. Generate gene targets with checkpoints
This section independently reloads the split and finalized genes. Choose start anew
or resume for both target files. `y_train.pt` and `y_test.pt` are overwritten every
500 patches with the completed prefix, plus a final save. Each has separate sidecars
for resuming. Partial tensors must not be used for training. Keep all sidecars.
Existing targets without these new sidecars cannot be resumed safely; start anew once.
If source CSV contents change, start anew for both statistics and targets.


In [ ]:
from pathlib import Path
import sys
_here = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (_here, *_here.parents) if (p / 'scripts').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from HEST_Local or its notebook folder.")
if str(PROJECT_ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

from pathlib import Path
import sys, importlib

HELPER_DIR = Path(globals().get('HELPER_DIR', str(PROJECT_ROOT / 'scripts')))
RUN_DIR = Path(globals().get('RUN_DIR', str(PROJECT_ROOT / 'training_run')))
sys.path.insert(0, str(HELPER_DIR))
import hest_content_pipeline as content
importlib.reload(content)
split = content.load_grouped_split(RUN_DIR)
train_records, test_records = split['train_records'], split['test_records']
records = split['records']
group_by_slide, grouping_unit = split['group_by_slide'], split['grouping_unit']

# Reload finalized genes from disk; section 5a need not run in this session.
selected_genes = content.load_selected_genes(RUN_DIR)
mode = content.prompt_checkpoint_mode('Gene target generation (train and test)')
for name, rows in [('train', train_records), ('test', test_records)]:
    content.save_target_tensor(
        rows, selected_genes, RUN_DIR/f'y_{name}.pt',
        checkpoint_every=500, checkpoint_mode=mode,
    )
    meta = content.validate_target_checkpoint(RUN_DIR/f'y_{name}.pt')
    print(f'{name} targets complete: {meta["shape"]}')
import torch


## 6. Extract PathFM embeddings with checkpoints
Choose start anew or resume before model loading. Default: save every 500 MODEL
BATCHES (1,000 patches at batch_size=2). Set EMBEDDING_CHECKPOINT_UNIT='patches'
to save every 500 patches instead. The final short interval is always saved.
Train and test have separate checkpoints. Saved embeddings retain manifest order.
Keep the .pt, .json, .checkpoint.json and both .npz snapshots on Drive.
A partial .pt contains only completed rows and cannot be used for PCA/training.
Resume requires the same split, batch size, interval and model/transform IDs.
Existing old embeddings without checkpoint sidecars need one fresh run.
If images, model weights, or transformations change, start anew.
NUM_WORKERS=4 loads/transforms images concurrently and prefetches the next batch.
Use 0 for serial loading. GPU inference stays ordered; output rows are unchanged.


In [ ]:
from pathlib import Path
import sys
_here = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (_here, *_here.parents) if (p / 'scripts').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from HEST_Local or its notebook folder.")
if str(PROJECT_ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

from pathlib import Path
import sys, importlib, json
import torch

HELPER_DIR = Path(globals().get('HELPER_DIR', str(PROJECT_ROOT / 'scripts')))
RUN_DIR = Path(globals().get('RUN_DIR', str(PROJECT_ROOT / 'training_run')))
sys.path.insert(0, str(HELPER_DIR))
import hest_content_pipeline as content
import hest_preprocessing as preprocessing
importlib.reload(content)
importlib.reload(preprocessing)
split = content.load_grouped_split(RUN_DIR)
train_records, test_records = split['train_records'], split['test_records']

BATCH_SIZE = 2
NUM_WORKERS = 4  # CPU image loaders; 0 disables parallel loading.
EMBEDDING_CHECKPOINT_EVERY = 500
EMBEDDING_CHECKPOINT_UNIT = 'batches'  # Use 'patches' for 500 patch rows per save.
mode = content.prompt_checkpoint_mode('PathFM embeddings (train and test)')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load PathFM once for unfinished work; skip weight loading if both are complete.
paths = [RUN_DIR/'embeddings_train.pt', RUN_DIR/'embeddings_test.pt']
completed = []
for path in paths:
    checkpoint = path.parent/f'{path.name}.checkpoint.json'
    try:
        done = mode == 'resume' and checkpoint.is_file() and json.loads(checkpoint.read_text()).get('complete', False)
    except json.JSONDecodeError:
        done = False  # The exporter will explain recovery; start anew ignores old state.
    completed.append(done)
pathfm, image_transform = None, None
if mode == 'new' or not all(completed):
    pathfm, image_transform = preprocessing.load_pathfm(device=device)

embeddings = {}
for name in ('train', 'test'):
    embeddings[name] = preprocessing.export_embeddings(
        RUN_DIR/f'{name}.csv', model=pathfm, transform=image_transform,
        output_path=RUN_DIR/f'embeddings_{name}.pt', batch_size=BATCH_SIZE, device=device,
        checkpoint_every=EMBEDDING_CHECKPOINT_EVERY,
        checkpoint_unit=EMBEDDING_CHECKPOINT_UNIT, checkpoint_mode=mode,
        num_workers=NUM_WORKERS,
        model_id='genbio-ai/genbio-pathfm', transform_id='pathfm_rgb224_norm_v1',
    )
    preprocessing.validate_embedding_checkpoint(
        RUN_DIR/f'embeddings_{name}.pt', RUN_DIR/f'{name}.csv')
X_train_raw, X_test_raw = embeddings['train'], embeddings['test']
print('Completed embeddings:', X_train_raw.shape, X_test_raw.shape)


## 7. Standardize and fit PCA on training embeddings only
StandardScaler and PCA both fit on training embeddings only; the held-out test embeddings are transformed with those fitted objects. PCA requires at least 250 training patches and embedding features.

In [ ]:
from pathlib import Path
import sys
_here = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (_here, *_here.parents) if (p / 'scripts').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from HEST_Local or its notebook folder.")
if str(PROJECT_ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

from pathlib import Path
import sys, importlib, json
import torch

HELPER_DIR = Path(globals().get('HELPER_DIR', str(PROJECT_ROOT / 'scripts')))
RUN_DIR = Path(globals().get('RUN_DIR', str(PROJECT_ROOT / 'training_run')))
sys.path.insert(0, str(HELPER_DIR))
import hest_content_pipeline as content
import hest_preprocessing as preprocessing
importlib.reload(content)
importlib.reload(preprocessing)
split = content.load_grouped_split(RUN_DIR)
train_records, test_records = split['train_records'], split['test_records']

# Restore completed tensors after a disconnect; never use a partial prefix.
for name in ('train', 'test'):
    preprocessing.validate_embedding_checkpoint(
        RUN_DIR/f'embeddings_{name}.pt', RUN_DIR/f'{name}.csv')
X_train_raw = torch.load(RUN_DIR/'embeddings_train.pt', map_location='cpu', weights_only=True)
X_test_raw = torch.load(RUN_DIR/'embeddings_test.pt', map_location='cpu', weights_only=True)
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import joblib
if X_train_raw.shape[0] < 250 or X_train_raw.shape[1] < 250:
    raise ValueError(f'PCA(250) needs at least 250 training patches and embedding features; got {tuple(X_train_raw.shape)}')
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw.numpy())
X_test_scaled = scaler.transform(X_test_raw.numpy())
pca = PCA(n_components=250, svd_solver='auto', random_state=42)
X_train = torch.from_numpy(pca.fit_transform(X_train_scaled).astype('float32'))
X_test = torch.from_numpy(pca.transform(X_test_scaled).astype('float32'))
torch.save(X_train, RUN_DIR/'X_train_pca250.pt')
torch.save(X_test, RUN_DIR/'X_test_pca250.pt')
joblib.dump(pca, RUN_DIR/'embedding_pca250.joblib')
joblib.dump(scaler, RUN_DIR/'embedding_scaler.joblib')
print('PCA output:', X_train.shape, X_test.shape,
      'variance retained:', float(pca.explained_variance_ratio_.sum()))

## 8a. Verify saved tensors
Edit four .pt paths in the cell. Shape checks cannot establish patch/gene identity or order.


In [ ]:
from pathlib import Path
import sys
_here = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (_here, *_here.parents) if (p / 'scripts').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from HEST_Local or its notebook folder.")
if str(PROJECT_ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

# Edit these paths to point to your tensors.
TRAIN_X = PROJECT_ROOT / 'training_run' / 'X_train_pca250.pt'
TRAIN_Y = PROJECT_ROOT / 'training_run' / 'y_train.pt'
TEST_X = PROJECT_ROOT / 'training_run' / 'X_test_pca250.pt'
TEST_Y = PROJECT_ROOT / 'training_run' / 'y_test.pt'
import importlib, pandas as pd
import hest_transformer_model as modeling
importlib.reload(modeling)
report = modeling.verify_training_tensors(TRAIN_X, TRAIN_Y, TEST_X, TEST_Y)
display(pd.DataFrame(report['files']))
print('Shape verification passed. Ensure matching patch and gene order.')


## Train from four tensor files
Edit the four tensor paths and parameters in the following cell. Verification is optional.


In [ ]:
from pathlib import Path
import sys
_here = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (_here, *_here.parents) if (p / 'scripts').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from HEST_Local or its notebook folder.")
if str(PROJECT_ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

# Edit tensor paths and training parameters directly.
TRAIN_X = PROJECT_ROOT / 'training_run' / 'X_train_pca250.pt'
TRAIN_Y = PROJECT_ROOT / 'training_run' / 'y_train.pt'
TEST_X = PROJECT_ROOT / 'training_run' / 'X_test_pca250.pt'
TEST_Y = PROJECT_ROOT / 'training_run' / 'y_test.pt'
OUTPUT_DIR = PROJECT_ROOT / 'training_run' / 'model'
EPOCHS = 30
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
D_MODEL = 64
N_HEADS = 4
N_LAYERS = 2
DROPOUT = 0.1
SEED = 42
DEVICE = None  # None automatically uses CUDA if available.
import importlib
import hest_transformer_model as modeling
importlib.reload(modeling)
model, metrics = modeling.train_transformer(
    TRAIN_X, TRAIN_Y, TEST_X, TEST_Y, output_dir=OUTPUT_DIR,
    epochs=EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
    d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS,
    dropout=DROPOUT, seed=SEED, device=DEVICE)
print('Final training MSE:', metrics['train_loss'])
print('Final test MSE:', metrics['test_loss'])
print('Mean test PCC:', metrics['mean_pcc'])
print('Mean std-normalized MAE:', metrics['mean_std_normalized_mae'])
print('Outputs:', OUTPUT_DIR)


## Outputs
`RUN_DIR` contains selected metadata, patient map, ordered train/test manifests, skipped-patch log, variance tables, top genes, embeddings, PCA tensors and fitted PCA, targets, model checkpoint, predictions and test metrics. Keep the JSON sidecars beside the embedding `.pt` files because they record patch ordering.